# Pan-tilt debug notebook

Talks directly to the positioner over serial using the PTCR-96 protocol (MN00162), bypassing `app.py` entirely (same pattern as `pantilt/pt_get_ping.py` in the original module).

**`app.py` must not be running at the same time** — a serial port can only be opened by one process. Stop the Flask app before running these cells, or stop the kernel before starting the app.

Run the first cell to connect, then run any of the other cells in any order/any number of times. Each cell prints the result so you can see immediately whether something failed.

Heater/max-speed/comm-timeout/identity commands aren't in `lib/qpt90.py` yet (their exact byte layout wasn't available when it was written) — those cells have been dropped for now.

In [1]:
# Cell 1 -- imports + connect
# Leave PORT_HINT = "" to auto-scan every detected serial port, or set it
# to a specific port (e.g. "COM5") to connect directly.

from lib import qpt90

PORT_HINT = "COM5"
BAUD = 9600

print(f"Connecting (port={PORT_HINT or 'auto-scan'}, baud={BAUD})...")
port, dev = qpt90.find_qpt90(port_hint=PORT_HINT, baud=BAUD)

if dev is None:
    print("FAILED: no pan-tilt responded. Candidate ports tried:", qpt90.list_candidate_ports())
else:
    print(f"Connected on {port}")

Connecting (port=COM5, baud=9600)...
Connected on COM5


In [2]:
# Cell 2 -- get status

status = dev.get_status()
print(f"pan={status.pan_deg:.2f} deg  tilt={status.tilt_deg:.2f} deg")
print(f"moving={status.moving}  hard_limit={status.hard_limit}  soft_limit={status.soft_limit}  continuous={status.continuous}")
print(f"faults: {', '.join(status.faults) if status.faults else 'none'}")

pan=6.05 deg  tilt=4.00 deg
moving=False  hard_limit=False  soft_limit=False  continuous=False
faults: none


In [7]:
# Cell 3 -- turn pan +1 deg (relative move)
import time

DELTA_DEG = 1.0
MOVE_TIMEOUT_S = 15

before = dev.get_status()
print(f"pan before: {before.pan_deg:.1f} deg")

dev.move_delta(pan_deg=DELTA_DEG, tilt_deg=0.0)

deadline = time.time() + MOVE_TIMEOUT_S
status = dev.get_status()
while time.time() < deadline and status.moving:
    time.sleep(0.2)
    status = dev.get_status()

print(f"pan after:  {status.pan_deg:.1f} deg  (moving={status.moving})")
print(f"faults: {', '.join(status.faults) if status.faults else 'none'}")

pan before: 10.1 deg
pan after:  11.1 deg  (moving=False)
faults: none


In [18]:
# Cell 4 -- turn pan -1 deg (relative move)
import time

DELTA_DEG = -1.0
MOVE_TIMEOUT_S = 15

before = dev.get_status()
print(f"pan before: {before.pan_deg:.1f} deg")

dev.move_delta(pan_deg=DELTA_DEG, tilt_deg=0.0)

deadline = time.time() + MOVE_TIMEOUT_S
status = dev.get_status()
while time.time() < deadline and status.moving:
    time.sleep(0.2)
    status = dev.get_status()

print(f"pan after:  {status.pan_deg:.1f} deg  (moving={status.moving})")
print(f"faults: {', '.join(status.faults) if status.faults else 'none'}")

pan before: 1.0 deg
pan after:  -0.0 deg  (moving=False)
faults: none


In [ ]:
# Cell 5 -- turn tilt +1 deg (relative move)
import time

DELTA_DEG = 1.0
MOVE_TIMEOUT_S = 15

before = dev.get_status()
print(f"tilt before: {before.tilt_deg:.1f} deg")

dev.move_delta(pan_deg=0.0, tilt_deg=DELTA_DEG)

deadline = time.time() + MOVE_TIMEOUT_S
status = dev.get_status()
while time.time() < deadline and status.moving:
    time.sleep(0.2)
    status = dev.get_status()

print(f"tilt after:  {status.tilt_deg:.1f} deg  (moving={status.moving})")
print(f"faults: {', '.join(status.faults) if status.faults else 'none'}")

In [22]:
# Cell 6 -- turn tilt -1 deg (relative move)
import time

DELTA_DEG = -1.0
MOVE_TIMEOUT_S = 15

before = dev.get_status()
print(f"tilt before: {before.tilt_deg:.1f} deg")

dev.move_delta(pan_deg=0.0, tilt_deg=DELTA_DEG)

deadline = time.time() + MOVE_TIMEOUT_S
status = dev.get_status()
while time.time() < deadline and status.moving:
    time.sleep(0.2)
    status = dev.get_status()

print(f"tilt after:  {status.tilt_deg:.1f} deg  (moving={status.moving})")
print(f"faults: {', '.join(status.faults) if status.faults else 'none'}")

tilt before: 1.0 deg
tilt after:  0.0 deg  (moving=False)
faults: none


In [ ]:
# Cell 7 -- move to an absolute pan/tilt position
# Edit the two target angles below, then run this cell.
import time

TARGET_PAN_DEG = 0.0
TARGET_TILT_DEG = 0.0
MOVE_TIMEOUT_S = 30

print(f"Moving to pan={TARGET_PAN_DEG} deg, tilt={TARGET_TILT_DEG} deg ...")
dev.move_to(pan_deg=TARGET_PAN_DEG, tilt_deg=TARGET_TILT_DEG)

deadline = time.time() + MOVE_TIMEOUT_S
status = dev.get_status()
while time.time() < deadline and status.moving:
    time.sleep(0.2)
    status = dev.get_status()

print(f"now at: pan={status.pan_deg:.1f} deg  tilt={status.tilt_deg:.1f} deg  (moving={status.moving})")
print(f"faults: {', '.join(status.faults) if status.faults else 'none'}")

In [ ]:
# Cell 8 -- stop motion

status = dev.stop()
print(f"Stopped. pan={status.pan_deg:.1f} deg  tilt={status.tilt_deg:.1f} deg  moving={status.moving}")

In [ ]:
# Cell 9 -- clear latched faults (TO/DE)

status = dev.clear_faults()
print(f"faults after clear: {', '.join(status.faults) if status.faults else 'none'}")

In [23]:
# Cell 10 -- close the serial connection (run this last, when you're done)

dev.ser.close()
print("Serial connection closed.")

Serial connection closed.
